In [ ]:
# =============================================================
# =============================================================
#
# recupero dati che si riferiscono ai comuni -- 
#
# =============================================================
# =============================================================

sorgente dati :https://situas.istat.it/web/#/apiDetail

Dettaglio API

Id report:
73
Titolo report:
Comuni - Caratteristiche del territorio
Inizio/fine validità report:
01/01/1948 - 10/05/2026
Analisi temporale:
DATA
Ambito territoriale:
Unità amministrative
Unità territoriale:
Comune
Parametri necessari:
pfun - pdata
pfun: Id report
pdata: data di estrazione (formato GG/MM/AAAA)
Dati:
https://situas-servizi.istat.it/publish/reportspooljson?pfun=73&pdata=01/01/1948
Conteggio:
https://situas-servizi.istat.it/publish/reportspooljsoncount?pfun=73&pdata=01/01/1948
Metadati:
https://situas-servizi.istat.it/publish/anagrafica_report_metadato_web?pfun=73&pdata=01/01/1948

In [4]:
# =============================================================
# =============================================================
#
# librerie di base
import numpy as np
import pandas as pd

In [5]:
# =============================================================
# =============================================================
#
# librerie fase recupero dati
import requests as api

In [6]:
# =============================================================
# =============================================================
#
# gestione file e folder e so
import os
import io
from io import StringIO
from os import listdir
from os.path import isfile, join

In [7]:
def GetFolderDati():
    folderdati = os.getcwd() 
    folderdati =folderdati+ "\\Data" 
    return folderdati

In [8]:
GetFolderDati()

'c:\\progetto_dataanalyst\\progettofinale\\Data'

In [14]:
# cancella il file se esiste
def CancellaFileSeEsiste(fullfilename: str):
     # verifico se esiste una precedente versione del file
    if os.path.exists(fullfilename):
        print(f"         Il file '{fullfilename}' esiste, per cui lo rimuovo")
        #cancello il file
        os.remove(fullfilename)
    else:
        print(f"      Il file '{fullfilename}' non esiste")

In [42]:
# salva il file nella cartella (estensione csv la inserisce la funzione)
def SalvaDataset(dati: DataFrame, nomefile: str):
    full_file_name = GetFolderDati() +  f'/{nomefile}.csv'

    print(f"      path completo file:  {full_file_name}")
    CancellaFileSeEsiste(full_file_name)
    
    #salvo il file perchè sarà usato in seguito per tutti i processi successivi
    dati.to_csv(full_file_name)
    print(f"      Il file salvato {full_file_name}")

In [18]:
# recupera dato da API e lo salva
def RecuperaESalvaDatiPerAnno(urldatiSenzaFiltroAnno : str, anno: int,prefisso_nome_file: int):
    #url completo
    urldati_anno = urldatiSenzaFiltroAnno  + "&pdata=01/01/" + str(anno)

    print('   Recupero  dati anno '+ str(anno))
    #eseguo la richiesta con Verbo Http GET e con l'aggiunta dell'header nella sezione della richiesta 
    rq_anno = api.get(urldati_anno)
    
    #lettura risposta
    status_code_risposta= rq_anno.status_code
    print (f"      status_code_risposta : {status_code_risposta}")
    if(status_code_risposta!=200):
        print(f'      errore richiesta api: status_code_risposta : {status_code_risposta}')
    else:
        print('      Converto i dati json ricevuti in DataFrame')
        df_anno = pd.DataFrame.from_dict(rq_anno.json()['resultset'])

        #aggiungo il riferimento del dataset --importantinssimo
        df_anno['anno_riferimento'] = anno

        nomefile=f"{prefisso_nome_file}_{str(anno)}" 
        print(f'      Salvo i dati in CSV su FS. Nome file={nomefile}')
        SalvaDataset(df_anno,nomefile)

In [19]:
def RecuperaESalvaDatiComune(anno: int):

    RecuperaESalvaDatiPerAnno("https://situas-servizi.istat.it/publish/reportspooljson?pfun=73",
                              anno,
                              "Dati_Comune")


In [20]:
#considero una finestra temporale di 10 anni.
print("Avvio esportazione dati comune per anno")
anno_esportazione= 2015
while anno_esportazione < 2026:
    print(f"Elaborazione dati comune anno {anno_esportazione}")
    RecuperaESalvaDatiComune(anno_esportazione)
    anno_esportazione= anno_esportazione+1

print("Fine esportazione dati comune per anno")

Avvio esportazione dati comune per anno
Elaborazione dati comune anno 2015
   Recupero  dati anno 2015
      status_code_risposta : 200
      Converto i dati json ricevuti in DataFrame
      Salvo i dati in CSV su FS. Nome file=Dati_Comune_2015
      path completo file:  c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2015.csv
         Il file 'c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2015.csv' esiste, per cui lo rimuovo
      Il file salvato c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2015.csv
Elaborazione dati comune anno 2016
   Recupero  dati anno 2016
      status_code_risposta : 200
      Converto i dati json ricevuti in DataFrame
      Salvo i dati in CSV su FS. Nome file=Dati_Comune_2016
      path completo file:  c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2016.csv
         Il file 'c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_2016.csv' esiste, per cui lo rimuovo
      Il file salvato c:\progetto_dataanalyst\progettofin

In [21]:
#aggregazione dataframe dei comuni
#per errore non lho fatto prima quando recuperavo il dataset per cui lo faccio ora
def UnioneOrizzontaleDataFrame(dataframe_a: DataFrame, dataframe_b: DataFrame):
    return pd.concat([dataframe_a,dataframe_b])

In [41]:

def UnioneFileCsvPerAnno(prefisso_file_Senza_Anno: str):
   folderdati = GetFolderDati()
   #print(f"Cartella dati: {folderdati}")  ok testato

   elencoelementi =os.listdir(folderdati)
   #print(elencoelementi)  ok testato
   elencofiledaconcatenare=[]
   for e in elencoelementi:
      #print(e)   ok testato
      fullfilename =os.path.join(folderdati, e)
      if(os.path.isfile(fullfilename)):
         #print("  e' un file")   ok testato
         if (e.startswith(f"{prefisso_file_Senza_Anno}_20")):
            elencofiledaconcatenare.append(fullfilename)

   #print(elencofiledaconcatenare)  ok testato
   #print (elencofiledaconcatenare[0]) ok testato

   #print(len(elencofiledaconcatenare)) ok testato
   risultato = pd.read_csv(elencofiledaconcatenare[0])
   for i in range(1, len(elencofiledaconcatenare)):
      #print(i) ok testato
      #print(elencofiledaconcatenare[i])    ok testato
      nuovofile = pd.read_csv(elencofiledaconcatenare[i])
      risultato = pd.concat([risultato,nuovofile])

   print(f"file: {prefisso_file_Senza_Anno}_All")
   SalvaDataset(risultato, f"{prefisso_file_Senza_Anno}_All")

In [43]:
UnioneFileCsvPerAnno("Dati_Comune")

file: Dati_Comune_All
      path completo file:  c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_All.csv
         Il file 'c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_All.csv' esiste, per cui lo rimuovo
      Il file salvato c:\progetto_dataanalyst\progettofinale\Data/Dati_Comune_All.csv
